# Now You Code In Class: 

## Tricks of The Pandas Masters Volume II

Once again, we will try something a bit different for our Activity - A series of Pandas coding challenges!

Datasets we will use:

- Reddit Data: https://raw.githubusercontent.com/mafudge/datasets/master/json-samples/reddit.json
- Episodes of the HBO series "The Wire": https://raw.githubusercontent.com/mafudge/datasets/master/tv-shows/the-wire.json


- Exam Scores: https://github.com/mafudge/datasets/blob/master/exam-scores/exam-scores.csv




In [5]:
import pandas as pd
import warnings

pd.set_option('display.max_colwidth', None)
warnings.filterwarnings('ignore')

## Deserializing json

the following function takes a URL as input and returns deserialized json as output.


In [6]:
def get_json(url: str):
    import requests
    response = requests.get(url)
    response.raise_for_status()
    return response.json()


This example deserializes reddit news and then finds the key where the articles are and displays the first artcle.

In [7]:
reddit = get_json("https://raw.githubusercontent.com/mafudge/datasets/master/json-samples/reddit.json")
articles = reddit["data"]["children"]
articles[0]

{'kind': 't3',
 'data': {'domain': 'wdrb.com',
  'banned_by': None,
  'media_embed': {},
  'subreddit': 'news',
  'selftext_html': None,
  'selftext': '',
  'likes': None,
  'suggested_sort': None,
  'user_reports': [],
  'secure_media': None,
  'link_flair_text': None,
  'id': '4bn12d',
  'from_kind': None,
  'gilded': 0,
  'archived': False,
  'clicked': False,
  'report_reasons': None,
  'author': 'homeboy422',
  'media': None,
  'score': 1,
  'approved_by': None,
  'over_18': False,
  'hidden': False,
  'num_comments': 1,
  'thumbnail': '',
  'subreddit_id': 't5_2qh3l',
  'hide_score': False,
  'edited': False,
  'link_flair_css_class': None,
  'author_flair_css_class': None,
  'downs': 0,
  'secure_media_embed': {},
  'saved': False,
  'removal_reason': None,
  'stickied': False,
  'from': None,
  'is_self': False,
  'from_id': None,
  'permalink': '/r/news/comments/4bn12d/man_arrested_for_stealing_suv_told_police_he/',
  'locked': False,
  'name': 't3_4bn12d',
  'created': 145877

In [8]:
# PROMPT 1 read in episodes of "The Wire" find the episodes key, and show the first episode.
thewire = get_json("https://raw.githubusercontent.com/mafudge/datasets/master/tv-shows/the-wire.json")
eps = thewire["_embedded"]["episodes"]
eps[0]

{'id': 12907,
 'url': 'https://www.tvmaze.com/episodes/12907/the-wire-1x01-the-target',
 'name': 'The Target',
 'season': 1,
 'number': 1,
 'type': 'regular',
 'airdate': '2002-06-02',
 'airtime': '21:00',
 'airstamp': '2002-06-03T01:00:00+00:00',
 'runtime': 60,
 'rating': {'average': 7.8},
 'image': {'medium': 'https://static.tvmaze.com/uploads/images/medium_landscape/94/236937.jpg',
  'original': 'https://static.tvmaze.com/uploads/images/original_untouched/94/236937.jpg'},
 'summary': '<p>Homicide detective Jimmy McNulty observes the murder trial of a mid-level drug dealer, D\'Angelo Barksdale, and sees the prosecution\'s star witness recant her testimony. McNulty recognises drug king-pin Stringer Bell in the court room and believes he has manipulated the proceedings, so he circumvents the chain-of-command by talking to the judge, Daniel Phelan, who then places pressure on the police department to investigate the Barksdale drug-dealing organization, which, McNulty claims, has gotten

## Creating dataframe

In this example we use`json_normalize()` to display the articles in a dataframe

In [9]:
art_df = pd.json_normalize(articles)
art_df.head(1)

,kind,data.domain,data.banned_by,data.subreddit,data.selftext_html,data.selftext,data.likes,data.suggested_sort,data.user_reports,data.secure_media,...,data.url,data.author_flair_text,data.quarantine,data.title,data.created_utc,data.distinguished,data.mod_reports,data.visited,data.num_reports,data.ups
0,t3,wdrb.com,None,news,None,,None,None,[],None,...,http://www.wdrb.com/story/31546565/man-arrested-for-stealing-suv-told-police-he-needed-it-for-job-interview,None,False,Man arrested for stealing SUV told police he needed it for job interview,1.458748e+09,None,[],False,None,1


In [10]:
# PROMPT 2 display the episodes as a dataframe
ep_df = pd.json_normalize(eps)
ep_df.head(1)

,id,url,name,season,number,type,airdate,airtime,airstamp,runtime,summary,rating.average,image.medium,image.original,_links.self.href
0,12907,https://www.tvmaze.com/episodes/12907/the-wire-1x01-the-target,The Target,1,1,regular,2002-06-02,21:00,2002-06-03T01:00:00+00:00,60,"<p>Homicide detective Jimmy McNulty observes the murder trial of a mid-level drug dealer, D'Angelo Barksdale, and sees the prosecution's star witness recant her testimony. McNulty recognises drug king-pin Stringer Bell in the court room and believes he has manipulated the proceedings, so he circumvents the chain-of-command by talking to the judge, Daniel Phelan, who then places pressure on the police department to investigate the Barksdale drug-dealing organization, which, McNulty claims, has gotten away with ten murders in the last year. D'Angelo is welcomed home by his uncle, Barksdale patriarch, Avon, who is frustrated with him for placing himself in a situation where the police could charge him. Nevertheless, Avon allows him to return to work, but in what D'Angelo sees as a demotion, he is moved to a low-rise housing project known as ""the pit."" Meanwhile, homeless drug addict Bubbles, acts as mentor to another addict, Johnny Weeks, in an ill-conceived scam with severe consequences.</p>",7.8,https://static.tvmaze.com/uploads/images/medium_landscape/94/236937.jpg,https://static.tvmaze.com/uploads/images/original_untouched/94/236937.jpg,https://api.tvmaze.com/episodes/12907


## A Text Sentiment Web Service

The following function uses the http://text-processing.com service to calculate the sentiment for any input text. The result is a dict with probabilities and overall sentiment. For example:

INPUT: `"very nice"`

OUTPUT: 
```
{
    'probability': {
        'neg': 0.28997418956645504,
        'neutral': 0.13591527211268692,
        'pos': 0.710025810433545
    },
    'label': 'pos'
}


In [11]:
def get_text_sentiment(text: str) -> dict:
    import requests
    response = requests.post("http://text-processing.com/api/sentiment/", data={"text": text})
    response.raise_for_status()
    return response.json()

sentiment = get_text_sentiment("very nice")
sentiment

HTTPError: 503 Server Error: SERVICE UNAVAILABLE for url: http://text-processing.com/api/sentiment/

In [ ]:
# PROMPT 3: get the sentiment of the text below and print the label only
text = "The Dallas Cowboys frustrate me. They have a great season but then tank in the playoffs."
sentiment = get_text_sentiment(text)
print(f"The sentiment is {sentiment['label']}")

## Using a lambda with `apply()`

The following example creates a new Series in the dataframe called `"title_sentiment"` which calculates the sentiment of the reddit news title.

In [12]:
art_df["title_sentiment"] = art_df.apply(lambda row: get_text_sentiment(row["data.title"])['label'], axis=1)
art_df[["data.title", "title_sentiment"]].head()

HTTPError: 503 Server Error: SERVICE UNAVAILABLE for url: http://text-processing.com/api/sentiment/

In [ ]:
# PROMPT 4 get the sentiment for the summary Series in the episodes dataframe, name the column "summary_sentiment" and show the summary and the sentiment
ep_df["summary_sentiment"] = ep_df.apply(lambda row: get_text_sentiment("summary")['label'], axis=1)
ep_df[["summary", "summary_sentiment"]].head()

## Saving the dataframe 

Calculating the sentiment isn't free and we don't need to do it more than once. Its a good idea to save the content of our dataframe at this point.  That way we can continue from the saved file versus re-calculating the transformation.

After you run this code open the `reddit_news_articles.csv` file and check it out!

In [13]:
art_df.to_csv("reddit_news_articles.csv", index=False)

In [14]:
# PROMPT 5 save your episodes of the wire to "the_wire_episodes.csv"
ep_df.to_csv("the_wire_episodes.csv", index=False)

## Hot and Top articles.

You have been hired as a data scientist at Reddit. Through your extensive statistical research, you have determined: 

- Articles where the number of comments are in the 75% percentile should be categorized as `"Hot"`
- Articles where the number of upvotes are in the 75% percentile should be categorized as `"Popular"`
- Articles matching both should be categories as `"Fire"`

In these next challenges, we will write code to implement these rules.


In [15]:
# PROMPT 6 write code to show the statistics of the numerical columns in the `art_df`
art_df.describe()

,data.gilded,data.score,data.num_comments,data.downs,data.created,data.created_utc,data.ups
count,25.0,25.000000,25.000000,25.0,2.500000e+01,2.500000e+01,25.000000
mean,0.0,30.440000,10.920000,0.0,1.458733e+09,1.458704e+09,30.440000
std,0.0,74.512348,22.597788,0.0,3.688108e+04,3.688108e+04,74.512348
min,0.0,0.000000,0.000000,0.0,1.458655e+09,1.458627e+09,0.000000
25%,0.0,1.000000,1.000000,0.0,1.458700e+09,1.458671e+09,1.000000
50%,0.0,8.000000,2.000000,0.0,1.458727e+09,1.458698e+09,8.000000
75%,0.0,27.000000,11.000000,0.0,1.458767e+09,1.458738e+09,27.000000
max,0.0,376.000000,108.000000,0.0,1.458791e+09,1.458762e+09,376.000000


In [16]:
# PROMPT 7 which columns represent comments? upvotes? What are the thresholds based on the 75% percentile?
# data.num_comments == 11
# data.ups == 27

In [17]:
# PROMPT 8 Write code to extract the comments_threshold
comments_threshold = art_df.describe().loc["75%", "data.num_comments"]
print(comments_threshold)

11.0


In [18]:
# PROMPT 9 Write code to extract the upvote_threshold
upvote_threshold = art_df.describe().loc["75%", "data.ups"]
print(upvote_threshold)

27.0


### defining the `categorize()` function:

Let's review the requirements: 

- Articles where the number of comments are in the 75% percentile should be categorized as "Hot"
- Articles where the number of upvotes are in the 75% percentile should be categorized as "Popular"
- Articles matching both should be categories as "Fire"

INPUTS (4): 

    PROMPT 10
    
OUTPUTS (1):

    PROMPT 11

In [19]:
# PROMPT 12 write function definition
def categorize(upvotes: float, upvote_threshold: float, comments: float, comments_threshold: float) -> str:
    pop = upvotes >= upvote_threshold
    hot = comments >= comments_threshold
    if pop and hot:
        return "Fire"
    if pop:
        return "Popular"
    if hot:
        return "Hot"
    return ""

In [26]:
# PROMPT 13a write test for "Hot"
upvotes = 10
upvote_threshold = 15
comments = 25
comments_threshold = 20
expect = "Hot"
actual = categorize(upvotes, upvote_threshold, comments, comments_threshold)
print(f"When upvotes={upvotes}/{upvote_threshold}, comments={comments}/{comments_threshold}, EXPECT={expect}, ACTUAL={actual}")
assert expect == actual

When upvotes=10/15, comments=25/20, EXPECT=Hot, ACTUAL=Hot


In [21]:
# PROMPT 13b write test for "Popular"
upvotes = 20
upvote_threshold = 15
comments = 10
comments_threshold = 20
expect = "Popular"
actual = categorize(upvotes, upvote_threshold, comments, comments_threshold)
print(f"When upvotes={upvotes}/{upvote_threshold}, comments={comments}/{comments_threshold}, EXPECT={expect}, ACTUAL={actual}")
assert expect == actual

When upvotes=20/15, comments=10/20, EXPECT=Popular, ACTUAL=Popular


In [22]:
# PROMPT 13c write test for "Fire"
upvotes = 20
upvote_threshold = 15
comments = 21
comments_threshold = 20
expect = "Fire"
actual = categorize(upvotes, upvote_threshold, comments, comments_threshold)
print(f"When upvotes={upvotes}/{upvote_threshold}, comments={comments}/{comments_threshold}, EXPECT={expect}, ACTUAL={actual}")
assert expect == actual

When upvotes=20/15, comments=21/20, EXPECT=Fire, ACTUAL=Fire


In [23]:
# PROMPT 13d write test for ""
upvotes = 2
upvote_threshold = 15
comments = 1
comments_threshold = 20
expect = ""
actual = categorize(upvotes, upvote_threshold, comments, comments_threshold)
print(f"When upvotes={upvotes}/{upvote_threshold}, comments={comments}/{comments_threshold}, EXPECT={expect}, ACTUAL={actual}")
assert expect == actual

When upvotes=2/15, comments=1/20, EXPECT=, ACTUAL=


## Final step. lambda / apply

Add a column to the dataframe called `"category"` which calculates the category for the article 

output the article title, the number of comments, number of upvotes and the category

In [24]:
# PROMPT 14
comments_threshold = art_df.describe().loc["75%", "data.num_comments"]
upvote_threshold = art_df.describe().loc["75%", "data.ups"]

art_df["category"] = art_df.apply(lambda row: categorize( row["data.ups"], upvote_threshold, row["data.num_comments"], comments_threshold), axis=1)
art_df[["data.title", "data.ups", "data.num_comments", "category"]]


,data.title,data.ups,data.num_comments,category
0,Man arrested for stealing SUV told police he needed it for job interview,1,1,
1,Zika outbreak needs $4 million: WHO,0,1,
2,"Austin police to fire officer who shot, killed nude teen",54,21,Fire
3,Hubble Unveils Monster Stars,11,0,
4,ISIS application forms surface in intelligence haul on terror group's recruits,1,0,
5,FDA Adds Boldest Warning To Most Widely Used Painkillers,17,2,
6,Brussels attacks expose gaping hole in airport security nets,3,5,
7,"Juveniles in Maryland's justice system are routinely strip-searched &amp; shackled: these practices apply to every youth...often for low-level offenses...also apply to juveniles who have not gone to court yet, and who could be found to have done nothing",376,108,Fire
8,Manhunt As Brussels Suspects Caught On CCTV: Two men who may be suicide bombers are wearing gloves that could have hidden detonators while a third man is being hunted.,38,2,Popular
9,Possible MH370 Debris Discovered in South Africa,2,0,


In [25]:
# run this code to turn in your work!
from casstools.assignment import Assignment
Assignment().submit()

✅ TIMESTAMP  : 2024-04-10 15:49
✅ COURSE     : ist256
✅ TERM       : spring2024
✅ USER       : mafudge@syr.edu
✅ STUDENT    : True
💣 ERROR GETTING ASSIGNMENT INFORMATION 💣
❌ Error Details: SOLUTION-SmallGroup-PandasII.ipynb is not an assignment on the course assignment list.
Possible Causes:
 - Is the assignment 'SOLUTION-SmallGroup-PandasII.ipynb'on the assignment list?
